# Runway Screening — Production Training on Colab (A100)

Trains **YOLOv8m** (fast v2 comparison run, ~2 h) on a merged corpus: RDD2022 + three top-down/runway transfer datasets (**UAV-PDD2023**, **HighRPD**, **FOD-A**) + the verified drone labels. This is the **v2** run — it adds top-down crack data and a real **fod** class.

The merge caps the bloated single-domain sources (FOD-A → 4k, RDD → 12k) so the corpus stays ~28k images and one class can't dominate training time.

**Before you run:** `Runtime → Change runtime type → A100 GPU` (Colab Pro/Pro+).
Run every cell top-to-bottom. Downloads total ~18 GB (RDD 13 + UAV-PDD 2.1 + HighRPD 1.7 + FOD-A 0.4); training is ~1.5–2.5 h. See cell 7 for the speed/precision dial.

## 1. Confirm the A100 is attached

In [ ]:
!nvidia-smi

## 2. Install ultralytics

In [ ]:
!pip -q install ultralytics
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 3. Clone the repo (branch `main`)
Brings in `src/`, the 274 drone `frames/`, and the 154 verified `drone_labels_v1/`.

In [ ]:
import os
REPO = '/content/runway_screening'
# Idempotent: pull latest if already cloned (a leftover dir must never shadow
# new code), clone fresh otherwise. Absolute path avoids nested re-clones.
if os.path.isdir(REPO + '/.git'):
    !git -C {REPO} pull origin main
else:
    !git clone --branch main --single-branch https://github.com/shabazahmadme1-eng/runway_screening.git {REPO}
%cd {REPO}

## 4. Download RDD2022 from FigShare (~13 GB, no auth)
Official CRDDC2022 release. `wget -c` resumes if the runtime hiccups.

In [ ]:
import os
os.makedirs('data/rdd2022', exist_ok=True)
!wget -c -O data/rdd2022/RDD2022_CRDDC.zip https://ndownloader.figshare.com/files/38030910

## 5. Extract (outer zip + the 7 nested per-country zips)
Uses Python `zipfile` — bsdtar chokes on this archive.

In [ ]:
import zipfile
from pathlib import Path

ZIP = Path('data/rdd2022/RDD2022_CRDDC.zip')
DEST = Path('data/rdd2022/extracted')
DEST.mkdir(parents=True, exist_ok=True)

print('extracting outer zip ...', flush=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DEST)
for nz in sorted(DEST.rglob('*.zip')):
    out = nz.parent / nz.stem
    print('extracting nested', nz.name, flush=True)
    with zipfile.ZipFile(nz) as z:
        z.extractall(out)
    nz.unlink()  # free disk as we go
print('xml:', sum(1 for _ in DEST.rglob('*.xml')), '| jpg:', sum(1 for _ in DEST.rglob('*.jpg')))

## 6. Build the master split (RDD2022 + verified drone labels)
Should report `nc: 12` and ~21k train / ~5.3k val.

## 5b. Download the extra transfer datasets
UAV-PDD2023 (Zenodo, VOC), HighRPD (Mendeley, YOLO), FOD-A (Drive, VOC). `wget -c` resumes on hiccups.

In [ ]:
import os, zipfile
from pathlib import Path

# UAV-PDD2023 (Zenodo, ~2.1 GB, PASCAL VOC, top-down 30 m)
os.makedirs('data/uav_pdd2023', exist_ok=True)
!wget -c -O data/uav_pdd2023/UAV-PDD2023.zip "https://zenodo.org/api/records/8429208/files/UAV-PDD2023.zip/content"
with zipfile.ZipFile('data/uav_pdd2023/UAV-PDD2023.zip') as z: z.extractall('data/uav_pdd2023')

# HighRPD (Mendeley, ~1.7 GB, YOLO, top-down 50 m)
os.makedirs('data/highrpd', exist_ok=True)
!wget -c -O data/highrpd/HighRPD.zip "https://data.mendeley.com/public-files/datasets/sywswj7djj/files/518cfb67-17f4-44a9-92a4-6d9d64cf8b83/file_downloaded"
with zipfile.ZipFile('data/highrpd/HighRPD.zip') as z: z.extractall('data/highrpd')

# FOD-A (Google Drive, ~412 MB, PASCAL VOC)
!pip -q install gdown
os.makedirs('data/fod_a', exist_ok=True)
!gdown 1RdErcq8PGRXZUOGauaACkQG44T-QyZ4x -O data/fod_a/FOD-A-VOC.zip
with zipfile.ZipFile('data/fod_a/FOD-A-VOC.zip') as z: z.extractall('data/fod_a')

for d in ['data/uav_pdd2023','data/highrpd','data/fod_a']:
    print(d, '| xml:', sum(1 for _ in Path(d).rglob('*.xml')),
          '| txt:', sum(1 for _ in Path(d).rglob('*.txt')),
          '| jpg:', sum(1 for _ in Path(d).rglob('*.jpg')))

In [ ]:
# rebuild from scratch so an earlier (uncapped) master can't leave stale images
!rm -rf datasets/master
!python src/merge_datasets.py --drone-frames frames --drone-labels drone_labels_v1 \
    --rdd-dir data/rdd2022/extracted --max-rdd-images 12000 \
    --uav-pdd-dir data/uav_pdd2023 --highrpd-dir data/highrpd \
    --fod-a-dir data/fod_a --max-fod-a 4000 \
    --out datasets/master
print('\n--- data.yaml ---')
print(open('datasets/master/data.yaml').read())

## 7. Train — YOLOv8m, fast v2 comparison run (~2 h)

You already have a working **v1** (`weights/yolov8m_v1.pt`, mAP50 0.693). The point of v2 is to test whether the added top-down crack data + the real `fod` class help — a ~2 h run answers that; a marathon isn't needed.

Recipe, and *why* (thin top-down runway cracks):
- **`--imgsz 768`** — still resolves cracks well; far cheaper than 1024/1280.
- **`--fraction 0.6`** — trains on 60% of the capped corpus each epoch (big speed-up, minor accuracy cost).
- **`--epochs 50 --patience 15`** — early-stops on plateau; usually settles well before 50.
- **`--flipud 0.5 --degrees 10`** — valid extra augmentation for top-down imagery.
- **`--cos-lr` / `--close-mosaic 10`** — smoother convergence, tighter final boxes.

Expect **~3 min/epoch → ~1.5–2.5 h total**, comfortably inside one session.

**Speed dial:** want it faster? lower `--imgsz` to 640 and `--fraction` to 0.4 (~45 min). Want max precision and have time? `--imgsz 1024 --fraction 1.0 --epochs 60` (~6–9 h). If you ever get disconnected mid-run, add `--project /content/drive/MyDrive/runway_runs` and re-run with `--resume`.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # tidy AutoBatch
!python src/train.py --model yolov8m.pt --data datasets/master/data.yaml \
    --epochs 50 --imgsz 768 --batch -1 --workers 8 \
    --cos-lr --close-mosaic 10 --flipud 0.5 --degrees 10 --box 7.5 \
    --fraction 0.6 --patience 15 --name runway_m_v2

## 8. Save artifacts to Google Drive
Copies `best.pt` (as `yolov8m_v2.pt`), `results.csv`, and `confusion_matrix.png` to `MyDrive/runway_m_v2/` so they survive the runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

run = 'runs/detect/runway_m_v2'
dst = '/content/drive/MyDrive/runway_m_v2'
os.makedirs(dst, exist_ok=True)
shutil.copy(f'{run}/weights/best.pt', f'{dst}/yolov8m_v2.pt')
for fn in ['results.csv', 'confusion_matrix.png', 'args.yaml']:
    p = f'{run}/{fn}'
    if os.path.exists(p):
        shutil.copy(p, dst)
print('saved to', dst, '->', sorted(os.listdir(dst)))
print('best.pt size MB:', round(os.path.getsize(f'{dst}/yolov8m_v2.pt') / 1e6, 1))

## 9. (Optional) Push artifacts straight back to GitHub
yolov8m `best.pt` is ~50 MB — under GitHub's 100 MB limit. Paste a [Personal Access Token](https://github.com/settings/tokens) (repo scope) and run. Otherwise skip and pull from Drive locally.

In [ ]:
TOKEN = ''  # <-- paste a GitHub PAT with 'repo' scope, then run this cell

if TOKEN:
    import os, shutil
    run = 'runs/detect/runway_m_v2'
    os.makedirs('weights', exist_ok=True)
    os.makedirs('reports/runway_m_v2', exist_ok=True)
    shutil.copy(f'{run}/weights/best.pt', 'weights/yolov8m_v2.pt')
    for fn in ['results.csv', 'confusion_matrix.png']:
        if os.path.exists(f'{run}/{fn}'):
            shutil.copy(f'{run}/{fn}', 'reports/runway_m_v2/')
    !git config user.email "neuroparadigm@gmail.com"
    !git config user.name "Shabaz"
    !git add -f weights/yolov8m_v2.pt reports/runway_m_v2
    !git -c commit.gpgsign=false commit -m "Add yolov8m production weights + reports (Colab A100, 960px)"
    !git remote set-url origin https://{TOKEN}@github.com/shabazahmadme1-eng/runway_screening.git
    !git push origin main
else:
    print('No token set — skipping push. Artifacts are in Drive from cell 8.')